In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
import geopandas as gpd
from shapely.geometry import Point
from sklearn.cluster import DBSCAN

In [2]:
df = pd.read_csv('C:\\Studia\\Master\\master_thesis\\data\\data1.csv', sep = ';')

In [3]:
pd.set_option('display.max_columns', None) 
pd.set_option('display.max_rows', None)  

df.head(5)



,?,ID,REGON,POWIAT,GMINA,Address_full,SEK_PKD7,PKD7,GR_LPRAC,lon,lat,tech,if_hightech,empl,sharesEMPL2,ones,locPdens,locAggA,locAggB,locAggC,locAggD,locAggE,locAggF,locAggG,locAggH,locAggI,locAggJ,locAggK,locAggL,locAggM,locAggN,locAggO,locAggP,locAggQ,locAggR,locAggS,locAggT,locAggU,locHH,locBIG,locAggTOTAL,locHightech,locLQ,dist_core_10,dist_core_25,dist_core_50,dist_midsize_10,dist_midsize_25,dist_midsize_50,dist_regional_10,dist_regional_25,dist_regional_50,dist_localbig_10,dist_localbig_25,dist_localbig_50,dist_localsmall_10,dist_localsmall_25,dist_localsmall_50,COREfirms,COREpopul
0,42342,1,1.406805e+13,17,44,Joachima Lelewela 27 05-480 Karczew,S,9499Z,1,21.23600,52.08347,NaN,0,5.034085,1.856614e-10,1,26,18,0,38,0,1,25,56,13,1,7,5,1,11,5,0,6,9,1,8,0,0,1.468536e-09,0,205,15,0.851179,0,1,1,0,0,0,0,0,1,1,1,1,0,1,1,1,1
1,42343,2,6.721748e+13,7,22,Rogożek 3 26-903 Rogożek,A,0150Z,1,21.36008,51.63636,NaN,0,5.035107,1.531587e-11,1,1,2,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2.994063e-11,0,3,0,2.586889,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,0,0
2,118240,3,1.109380e+12,65,58,Ignacego Krasickiego 12A 02-611 Warszawa,G,4690Z,1,21.02140,52.19191,NaN,0,5.001276,1.518227e-11,1,203,72,4,141,24,12,192,620,111,64,203,98,185,533,121,0,89,101,51,207,0,1,2.685278e-07,34,2829,183,1.126141,1,1,1,0,0,0,0,1,1,0,1,1,1,1,1,1,1
3,210007,4,1.566525e+12,65,188,al. Prymasa Tysiąclecia 62 01-424 Warszawa,G,4643Z,1,20.95726,52.23474,NaN,0,5.114140,1.587524e-11,1,220,10,1,35,3,4,47,133,25,27,42,19,54,111,29,3,25,24,12,38,0,0,3.089481e-08,6,642,49,1.064512,1,1,1,0,0,0,0,1,1,0,1,1,0,1,1,1,1
4,423467,5,1.401995e+13,65,48,Marymoncka 93/97 01-813 Warszawa,S,9511Z,1,20.95403,52.28521,NaN,0,4.939015,1.787151e-10,1,150,50,0,172,2,2,166,505,110,59,143,66,105,323,104,1,106,111,69,164,0,0,5.495024e-07,16,2258,141,1.584181,1,1,1,0,0,0,0,1,1,1,1,1,1,1,1,1,1


In [ ]:
df.info() 

In [ ]:
df['SEK_PKD7'].value_counts()

In [68]:
df[['SEK_PKD7', 'if_hightech']].query('if_hightech == 1').groupby('SEK_PKD7').count().sort_values(by = 'if_hightech', ascending = False)

,if_hightech
SEK_PKD7,
J,33366
C,8020
M,955


In [6]:
sek_fin_mask = df['SEK_PKD7'] == 'H'
df['if_fin'] = np.where(sek_fin_mask, 1, 0)

In [30]:
# df[['SEK_PKD7', 'if_hightech', 'PKD7']].query('if_hightech == 1 and SEK_PKD7 == C')
df[(df['if_hightech'] == 1) & (df['SEK_PKD7'] == 'C')][['POWIAT', 'PKD7']].groupby(['POWIAT']).count()

,PKD7
POWIAT,
1,16
2,64
3,63
4,23
5,148
6,85
7,31
8,163
9,19


In [38]:
df[df['if_sek_fin'] == 1][['POWIAT', 'if_hightech']].groupby(['POWIAT']).count().apply(lambda x: x.sort_values(ascending=False))


,if_hightech
POWIAT,
65,20560
34,2113
21,1730
63,1666
18,1465
32,1236
28,1153
62,1006
8,955


In [41]:
df[(df['if_sek_fin'] == 1)][['if_sek_fin', 'PKD7']].groupby(['PKD7']).count().apply(lambda x: x.sort_values(ascending=False))


,if_sek_fin
PKD7,
4941Z,24387
4932Z,12693
4939Z,1869
5221Z,1339
5229C,1217
5320Z,1119
5210B,542
4931Z,379
5223Z,269


In [ ]:
df[['SEK_PKD7', 'if_sek_fin', 'if_hightech']].groupby(['if_sek_fin', 'if_hightech']).count()

In [43]:
kod_tech = pd.read_csv("C:\\Studia\\Master\\master_thesis\\data\\lista high tech.csv", sep = ";")

In [44]:
pop_point = pd.read_csv("C:\\Studia\\Master\\master_thesis\\data\\points_popul_maz.csv", sep = ",")

In [52]:
kod_tech_mapping = kod_tech.set_index('kod')['tech']

df['techSPEC'] = df['PKD7'].map(kod_tech_mapping)

df_copy = df.copy()

df_copy['techSPEC'] = df_copy['techSPEC'].fillna('other')

df_copy['techSPEC'].value_counts()

other                         941378
high-tech know-intens serv     34321
medium-high-tech                5847
high-tech                       2173
Name: techSPEC, dtype: int64

In [45]:
vars_list = ['COREfirms',   'COREpopul',   'dist_core_10',   'dist_core_25',	'dist_core_50',	'dist_midsize_10',	'dist_midsize_25',	'dist_midsize_50',
             'dist_regional_10',	'dist_regional_25',	'dist_regional_50',	'dist_localbig_10',	'dist_localbig_25',	'dist_localbig_50',
             'dist_localsmall_10','dist_localsmall_25', 'dist_localsmall_50']


for var in vars_list:

    cross_tab = pd.crosstab(df['if_hightech'], df[var])
    non_other_obs = cross_tab.index != 'other'
    matching_obs = cross_tab[1][non_other_obs].sum()
    total_non_other_obs = cross_tab[1][non_other_obs].sum() + cross_tab[0][non_other_obs].sum()
    percentage_matching = (matching_obs / total_non_other_obs) * 100

    print(f"For variable '{var}', percentage of high-tech observations from 'techSPEC' that are inside the radius of '{var}': {percentage_matching:.2f}%")
    print(cross_tab)
    print("-" * 50)

For variable 'COREfirms', percentage of high-tech observations from 'techSPEC' that are inside the radius of 'COREfirms': 69.22%
COREfirms         0       1
if_hightech                
0            300455  640923
1              2379   39962
--------------------------------------------------
For variable 'COREpopul', percentage of high-tech observations from 'techSPEC' that are inside the radius of 'COREpopul': 75.17%
COREpopul         0       1
if_hightech                
0            242915  698463
1              1333   41008
--------------------------------------------------
For variable 'dist_core_10', percentage of high-tech observations from 'techSPEC' that are inside the radius of 'dist_core_10': 33.40%
dist_core_10       0       1
if_hightech                 
0             636719  304659
1              18402   23939
--------------------------------------------------
For variable 'dist_core_25', percentage of high-tech observations from 'techSPEC' that are inside the radius of 'd

In [74]:
vars_list = ['COREfirms', 'COREpopul', 'dist_core_10', 'dist_core_25', 'dist_core_50', 'dist_midsize_10', 'dist_midsize_25', 'dist_midsize_50',
             'dist_regional_10', 'dist_regional_25', 'dist_regional_50', 'dist_localbig_10', 'dist_localbig_25', 'dist_localbig_50',
             'dist_localsmall_10', 'dist_localsmall_25', 'dist_localsmall_50']

results = []

for var in vars_list:
    cross_tab = pd.crosstab(df_copy['techSPEC'], df[var])
    non_other_obs = cross_tab.index != 'other'
    matching_obs = cross_tab[1][non_other_obs].sum()
    total_non_other_obs = cross_tab[1][non_other_obs].sum() + cross_tab[0][non_other_obs].sum()
    percentage_hightech = (matching_obs / total_non_other_obs) * 100
    percentage_all = (matching_obs / df.shape[0]) * 100


    result_dict = {
        'Variable': var,
        'Percentage hhightech': f"{percentage_hightech:.2f}%",
        'Percentage All': f"{percentage_all:.2f}%",
        'High-tech (0)': cross_tab.iloc[0,0],
        'High-tech (1)': cross_tab.iloc[0,1],
        'High-tech know-intens (0)': cross_tab.iloc[1,0],
        'High-tech know-intens (1)': cross_tab.iloc[1,1],
        'Medium high-tech (0)': cross_tab.iloc[2,0],
        'Medium high-tech (1)': cross_tab.iloc[2,1],
        'Non-high/medium-tech (0)': cross_tab.iloc[3,0],
        'Non-high/medium-tech (1)': cross_tab.iloc[3,1]

    }
    display(cross_tab)

    results.append(result_dict)

# Creating a DataFrame from the results list
results_tech_df = pd.DataFrame(results)

# display(results_tech_df)
# display(cross_tab)
# results_df.to_excel("crosstab.xlsx")

COREfirms,0,1
techSPEC,,
high-tech,107,2066
high-tech know-intens serv,1612,32709
medium-high-tech,660,5187
other,300455,640923


COREpopul,0,1
techSPEC,,
high-tech,48,2125
high-tech know-intens serv,889,33432
medium-high-tech,396,5451
other,242915,698463


dist_core_10,0,1
techSPEC,,
high-tech,980,1193
high-tech know-intens serv,13883,20438
medium-high-tech,3539,2308
other,636719,304659


dist_core_25,0,1
techSPEC,,
high-tech,301,1872
high-tech know-intens serv,4514,29807
medium-high-tech,1786,4061
other,457343,484035


dist_core_50,0,1
techSPEC,,
high-tech,174,1999
high-tech know-intens serv,2655,31666
medium-high-tech,1221,4626
other,345968,595410


...

dist_localbig_25,0,1
techSPEC,,
high-tech,154,2019
high-tech know-intens serv,2567,31754
medium-high-tech,1163,4684
other,327510,613868


dist_localbig_50,0,1
techSPEC,,
high-tech,97,2076
high-tech know-intens serv,1388,32933
medium-high-tech,643,5204
other,178933,762445


dist_localsmall_10,0,1
techSPEC,,
high-tech,480,1693
high-tech know-intens serv,7068,27253
medium-high-tech,1886,3961
other,439298,502080


dist_localsmall_25,0,1
techSPEC,,
high-tech,57,2116
high-tech know-intens serv,1185,33136
medium-high-tech,504,5343
other,156567,784811


dist_localsmall_50,0,1
techSPEC,,
high-tech,1,2172
high-tech know-intens serv,29,34292
medium-high-tech,12,5835
other,10550,930828


In [79]:
vars_list = ['COREfirms', 'COREpopul', 'dist_core_10', 'dist_core_25', 'dist_core_50', 'dist_midsize_10', 'dist_midsize_25', 'dist_midsize_50',
             'dist_regional_10', 'dist_regional_25', 'dist_regional_50', 'dist_localbig_10', 'dist_localbig_25', 'dist_localbig_50',
             'dist_localsmall_10', 'dist_localsmall_25', 'dist_localsmall_50']

results = []

for var in vars_list:
    cross_tab = pd.crosstab(df['if_sek_fin'], df[var])
    non_other_obs = cross_tab.index != 0
    matching_obs = cross_tab[1][non_other_obs].sum()
    print(cross_tab[1][non_other_obs])
    total_non_other_obs = cross_tab[1][non_other_obs].sum() + cross_tab[0][non_other_obs].sum()
    print(cross_tab[1][non_other_obs].sum() + cross_tab[0][non_other_obs].sum())
    percentage_matching = (matching_obs / total_non_other_obs) * 100
    percentage_all = (matching_obs / df.shape[0]) * 100

    result_dict = {
        'Variable': var,
        'Percentage Matching': f"{percentage_matching:.2f}%",
        'Percentage All': f"{percentage_all:.2f}%",
        'is_fin (0)': cross_tab.iloc[0, 0],
        'is_fin (1)': cross_tab.iloc[0, 1],
    }

    results.append(result_dict)
    display(cross_tab)
# Creating a DataFrame from the results list
results_fin_df = pd.DataFrame(results)

display(results_fin_df)
display(cross_tab)


if_sek_fin
1    36844
Name: 1, dtype: int64
44894


COREfirms,0,1
if_sek_fin,,
0,294784,644041
1,8050,36844


if_sek_fin
1    39252
Name: 1, dtype: int64
44894


COREpopul,0,1
if_sek_fin,,
0,238606,700219
1,5642,39252


if_sek_fin
1    16678
Name: 1, dtype: int64
44894


...

dist_localsmall_25,0,1
if_sek_fin,,
0,152481,786344
1,5832,39062


if_sek_fin
1    44694
Name: 1, dtype: int64
44894


dist_localsmall_50,0,1
if_sek_fin,,
0,10392,928433
1,200,44694


,Variable,Percentage Matching,Percentage All,is_fin (0),is_fin (1)
0,COREfirms,82.07%,3.75%,294784,644041
1,COREpopul,87.43%,3.99%,238606,700219
2,dist_core_10,37.15%,1.70%,626905,311920
3,dist_core_25,60.69%,2.77%,446298,492527
4,dist_core_50,72.47%,3.31%,337659,601166
5,dist_midsize_10,4.13%,0.19%,910054,28771
6,dist_midsize_25,8.90%,0.41%,859177,79648
7,dist_midsize_50,12.91%,0.59%,788354,150471
8,dist_regional_10,9.79%,0.45%,865993,72832
9,dist_regional_25,61.73%,2.82%,445939,492886


dist_localsmall_50,0,1
if_sek_fin,,
0,10392,928433
1,200,44694


In [ ]:
clustering_high_tech = df_copy[df_copy['techSPEC'] != 'other'][['lon', 'lat']]

geometry = [Point(xy) for xy in zip(clustering_high_tech['lon'], clustering_high_tech['lat'])]
gdf = gpd.GeoDataFrame(clustering_high_tech, geometry=geometry, crs="EPSG:4326")

# Perform DBSCAN clustering
eps = 0.11  # Maximum distance between points to consider them in the same neighborhood
min_samples = 75  # Minimum number of points in a neighborhood to consider it a core point
dbscan = DBSCAN(eps=eps, min_samples=min_samples)
gdf['cluster'] = dbscan.fit_predict(clustering_high_tech)

# Create subplots
fig, ax = plt.subplots(figsize=(10, 10))

# Plot clusters
gdf.plot(column='cluster', categorical=True, legend=True, markersize=3, ax=ax, cmap='tab20', missing_kwds={'color': 'grey'})

# Display the plot
plt.show()


In [ ]:
geometry = [Point(xy) for xy in zip(df['lon'], df['lat'])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")


fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(20, 10))

gdf.plot(column='locPdens', scheme='quantiles', ax=axes[0], legend=True, markersize=3, legend_kwds={'bbox_to_anchor': (0.7, 1)})
axes[0].set_title('locPdens')

gdf.plot(column='locAggTOTAL', scheme='quantiles', ax=axes[1], legend=True, markersize=3, legend_kwds={'bbox_to_anchor': (0.65, 1)})
axes[1].set_title('locAggTOTAL')

gdf.plot(column='locBIG', scheme='quantiles', ax=axes[2], legend=True, markersize=3, legend_kwds={'bbox_to_anchor': (0.7, 1)})
axes[2].set_title('locBIG')

plt.tight_layout()
plt.show()

In [ ]:
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt

geometry = [Point(xy) for xy in zip(df['lon'], df['lat'])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

# center_point = Point(21.0067, 52.2370)

# # Create a buffer of 10, 25, and 50 kilometers around the center point
# buffer_10km = center_point.buffer(0.1)  # 0.1 degrees approx. 10 km at the equator
# buffer_25km = center_point.buffer(0.25)  # 0.25 degrees approx. 25 km at the equator
# buffer_50km = center_point.buffer(0.5)  # 0.5 degrees approx. 50 km at the equator

# Plot the GeoDataFrame and circles
fig, ax = plt.subplots(figsize=(10, 5))
gdf.plot(column='techSPEC', ax=ax, legend=True, markersize=2, legend_kwds={'bbox_to_anchor': (0.64, 0.82)})

# Plot circles around the center point
# gpd.GeoSeries(buffer_10km).plot(ax=ax, color='none', edgecolor='black', linewidth=2)
# gpd.GeoSeries(buffer_25km).plot(ax=ax, color='none', edgecolor='black', linewidth=2)
# gpd.GeoSeries(buffer_50km).plot(ax=ax, color='none', edgecolor='black', linewidth=2)

plt.tight_layout()
plt.show()


In [ ]:
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt

# Your DataFrame df and its 'lon' and 'lat' columns should be defined here

geometry = [Point(xy) for xy in zip(df['lon'], df['lat'])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

# Define the center point
center_point = Point(21.0067, 52.2370)

# Convert the buffer sizes to the plot's coordinate system (assuming it's in meters)
buffer_size_meters = [10000, 25000, 50000]  # 10 km, 25 km, 50 km

# Create buffers using the buffer sizes in meters
buffers = [center_point.buffer(buffer_size / (40008000 / 360)) for buffer_size in buffer_size_meters]

# Convert the buffers to the same coordinate system as gdf
buffers = gpd.GeoSeries(buffers, crs="EPSG:4326")
buffers = buffers.to_crs(gdf.crs)

# Plot the GeoDataFrame and circles
fig, ax = plt.subplots(figsize=(10, 5))
gdf.plot(column='techSPEC', ax=ax, legend=True, markersize=2, legend_kwds={'bbox_to_anchor': (0.64, 0.82)})

# Plot circles around the center point
buffers.plot(ax=ax, color='none', edgecolor='black', linewidth=2)

plt.tight_layout()
plt.show()


In [ ]:
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import pandas as pd

# Assuming gdf is your GeoDataFrame
center_lon, center_lat = 21.0067, 52.2370  # Center of Warsaw
center_point = Point(center_lon, center_lat)

# Calculate accurate distances using geodesic distance
buffer_10km = center_point.buffer(0.1)  # 0.1 degrees approx. 10 km at the equator
buffer_25km = center_point.buffer(0.25)  # 0.25 degrees approx. 25 km at the equator
buffer_50km = center_point.buffer(0.5)


# Assuming you have calculated percentage_matching for each radius
percentage_matching_10km = 56.54  # Replace with actual calculated percentage
percentage_matching_25km = 84.41  # Replace with actual calculated percentage
percentage_matching_50km = 90.43  # Replace with actual calculated percentage

# Create GeoSeries for the circles
circle_10km = gpd.GeoSeries(buffer_10km)
circle_25km = gpd.GeoSeries(buffer_25km)
circle_50km = gpd.GeoSeries(buffer_50km)

# Plot the GeoDataFrame and circles
fig, ax = plt.subplots(figsize=(20,6))
gdf.plot(column='techSPEC', ax=ax, legend=True, markersize=2, legend_kwds={'bbox_to_anchor': (1, 1)})

# Plot circles around the center point with more visible lines
circle_10km.plot(ax=ax, color='none', edgecolor='blue', linewidth=2, linestyle='solid')
circle_25km.plot(ax=ax, color='none', edgecolor='yellow', linewidth=2, linestyle='solid')
circle_50km.plot(ax=ax, color='none', edgecolor='red', linewidth=2, linestyle='solid')

# Create legend handles for the circles
legend_handles = [
    Patch(color='blue', edgecolor='blue', linewidth=2, linestyle='solid', label=f'10km:{percentage_matching_10km:.2f}%'),
    Patch(color='yellow', edgecolor='yellow', linewidth=2, linestyle='solid', label=f'25km:{percentage_matching_25km:.2f}%'),
    Patch(color='red', edgecolor='red', linewidth=2, linestyle='solid', label=f'50km:{percentage_matching_50km:.2f}%')
]

# Add legend with custom handles
ax.legend(handles=legend_handles)

plt.show()


In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(20, 10))


gdf.plot(column='techSPEC', ax=axes[0], legend=True, markersize=5, legend_kwds={'bbox_to_anchor': (1, 1)})
axes[0].set_title('techSPEC')

gdf.plot(column='empl', scheme='quantiles', ax=axes[1], legend=True, markersize=5, legend_kwds={'bbox_to_anchor': (1, 1)})
axes[1].set_title('empl')

gdf.plot(column='locBIG', scheme='quantiles', ax=axes[2], legend=True, markersize=5, legend_kwds={'bbox_to_anchor': (1, 1)})
axes[2].set_title('locBIG')

plt.tight_layout()
plt.show()